# 🎯 11장 캡스톤 — 7B는 왜 4GB인가 🎓

졸업 시험입니다 — 새 개념 없이, 배운 전부로 **모델 영수증을 완독**하고 **KV 캐시까지
정산**해 내 PC 판정을 내립니다. 마지막엔 (Ollama 가 있다면) **실제 설치된 모델**과
우리 계산을 대조합니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter` — 위에서 아래로.

**이 노트북의 목표**: 워밍업의 질문에 스스로 답하기 —
**"개수(1장) × 칸(3~9장) = 파일, 파일 + 기억 장부(KV) + 고정 몫 = 실행."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch11')

## 1. 모델 영수증 완독기 — 전 챕터 총조립

In [ ]:
def count_params(V, d, L):
    """1장의 몸무게 공식 — 임베딩 V·d + 블록 L·12d²."""
    return V * d + L * 12 * d**2

SPECS = {'1B급': (32000, 2048, 16), '7B급': (32000, 4096, 32), '13B급': (32000, 5120, 40)}
TAGS = {'F16': 16.0, 'Q8_0': 8.5, 'Q4_K_M': 4.85, 'Q2_K': 2.6}   # 실효 비트 (7~9장)

def weights_gb(spec, tag):
    n = count_params(*SPECS[spec])
    return n * TAGS[tag] / 8 / 1e9

print(' 모델    파라미터        F16      Q8_0    Q4_K_M    Q2_K')
for name, s in SPECS.items():
    n = count_params(*s)
    row = '  '.join(f'{weights_gb(name, t):6.1f}' for t in TAGS)
    print(f' {name:<5} {n/1e9:6.2f}B      {row}')

assert abs(weights_gb('7B급', 'Q4_K_M') - 3.98) < 0.05, '🎓 워밍업의 답 — 7B × 4.85bit ÷ 8 ≈ 4.0GB'
assert abs(weights_gb('7B급', 'F16') - 13.1) < 0.1, '13GB 파일의 정체(7장)'
assert abs(weights_gb('13B급', 'Q4_K_M') - 7.7) < 0.2, '13B 도 q4 면 8GB 급'
print()
print('✔ 확인: 영수증 = 개수(1장 공식) × 실효 비트(3~9장) ÷ 8 — 전부 손으로 계산된다')

## 2. 메모리 정산기 — KV 캐시(대화의 기억 장부)까지

실행 = 가중치 + KV 캐시 + OS·실행기 고정 몫. KV 어림 공식(고전 멀티헤드 기준):
**토큰당 = 2(K·V) × 층수 L × 차원 d × 2바이트(fp16)** — 4부의 그 K·V 입니다.

In [ ]:
FIXED_GB = 1.5   # OS·실행기 고정 몫 (어림)

def kv_per_token_bytes(spec):
    V, d, L = SPECS[spec]
    return 2 * L * d * 2

def total_gb(spec, tag, ctx_tokens):
    return weights_gb(spec, tag) + kv_per_token_bytes(spec) * ctx_tokens / 1e9 + FIXED_GB

def verdict(spec, tag, ctx, ram):
    t = total_gb(spec, tag, ctx)
    state = '❌ 불가' if t > ram else ('⚠️ 아슬' if t > ram * 0.85 else '✅ 여유')
    return t, state

print(f'7B급 KV = {kv_per_token_bytes("7B급")/1e6:.2f}MB/토큰')
print(' 시나리오 (7B급 · Q4_K_M)                총 필요     판정')
for ctx, ram in [(4096, 8), (8192, 8), (4096, 16), (32768, 16)]:
    t, s = verdict('7B급', 'Q4_K_M', ctx, ram)
    print(f'  문맥 {ctx:>6,} 토큰 · RAM {ram:>2}GB      {t:5.1f} GB     {s}')

assert kv_per_token_bytes('7B급') == 524288, '7B급 KV = 0.5MB/토큰 (2×32×4096×2)'
t4k, s4k = verdict('7B급', 'Q4_K_M', 4096, 8)
t8k, s8k = verdict('7B급', 'Q4_K_M', 8192, 8)
assert s4k == '⚠️ 아슬' and s8k == '❌ 불가', '같은 파일인데 문맥 4K(아슬) → 8K(불가) — 8GB PC 가 무너진다'
assert verdict('7B급', 'Q4_K_M', 32768, 16)[1] == '❌ 불가', '32K 문맥은 KV 만 17GB — 긴 문맥이 비싼 이유'
print()
print('✔ 확인: 파일이 들어간다 ≠ 돌아간다 — 어림 규칙(10장)의 4GB 여유가 이 몫이었다')

## 3. 훈련의 몸무게 — 추론과의 격차 (7장 회수)

In [ ]:
# 혼합 정밀도 훈련(7장): fp16 사본 2B + fp32 원본 4B + Adam m·v 8B + gradient 2B ≈ 16B/파라미터
TRAIN_BYTES_PER_PARAM = 16
n7 = count_params(*SPECS['7B급'])
train_gb = n7 * TRAIN_BYTES_PER_PARAM / 1e9
infer_gb = weights_gb('7B급', 'Q4_K_M')
print(f'7B 훈련(혼합 정밀도) ≈ {train_gb:.0f} GB   vs   추론(Q4_K_M) ≈ {infer_gb:.1f} GB   ({train_gb/infer_gb:.0f}배)')

assert train_gb > 100 and train_gb / infer_gb > 20, '파인튜닝은 클라우드, 추론은 내 PC — 구도의 산수'
print()
print('✔ 확인: LoRA(6부 8장)가 사랑받는 이유도 이 표 안에 있다 — 훈련 장부를 저차원으로 줄이는 것')

## 4. (선택) 실전 대조 — 내 Ollama 의 실제 모델과 비교

Ollama 가 실행 중이면 설치된 모델들의 **실제 파일 크기**를 읽어 우리 영수증과 대조합니다.
없어도 괜찮아요 — 안내만 출력하고 넘어갑니다(학습 지장 없음).

In [ ]:
import requests

try:
    res = requests.get('http://localhost:11434/api/tags', timeout=3)
    models = res.json().get('models', [])
    if not models:
        print('Ollama 는 떠 있는데 설치된 모델이 없어요 — ollama pull 후 다시 실행해 보세요.')
    else:
        print(' 설치된 모델                          실제 크기      (우리 식으로 읽기)')
        for m in models[:10]:
            size_gb = m.get('size', 0) / 1e9
            details = m.get('details', {})
            q = details.get('quantization_level', '?')
            p = details.get('parameter_size', '?')
            print(f' {m["name"]:<36} {size_gb:6.1f} GB    {p} × {q} — 개수 × 칸!')
        print()
        print('✔ 태그·크기가 전부 읽히나요? 그게 9부의 졸업입니다 🎓')
except requests.RequestException:
    print('Ollama 미실행 — 건너뜁니다. (실행해 두면 실제 모델 크기와 영수증을 대조할 수 있어요)')

**직접 해보기**: ① `SPECS` 에 70B급(d=8192, L=80)을 추가해 Q4_K_M 크기를 계산해 보세요
(≈42GB — 내 PC 상한의 근거). ② GQA 를 흉내 내 `kv_per_token_bytes` 를 4로 나눠 보세요 —
같은 16GB 로 문맥이 몇 배 길어지나요?

## 5. 🎓 졸업 — 9부 수료

- **워밍업의 답**: 7B(= V·d + L·12d², 1장) × 실효 4.85비트(3~9장) ÷ 8 = **4.0GB**(9장),
  품질 대가 +1% 급(10장 무릎 위).
- **실행의 답**: + KV 캐시(토큰당 2×L×d×2B — 7B급 0.5MB/토큰) + 고정 몫 →
  8GB PC 에서 4K 문맥 아슬, 8K 는 불가 — **파일이 들어간다 ≠ 돌아간다.**
- **훈련의 답**: 파라미터당 ≈16B → 7B 훈련 105GB = 추론의 25배 — 파인튜닝은 클라우드,
  추론은 내 PC(LoRA 의 존재 이유).
- 이 노트북 옆의 🤖 튜터가 바로 오늘 완독한 **양자화 GGUF** 위에서 돌고 있습니다 —
  비용 0·데이터 유출 0 의 수학.

**시리즈의 세 부작 완성** — 6부 "무엇을 배우게 할까" · 8부 "어떻게 잘 배우게 할까" ·
9부 "**무엇으로 담고 계산하나**". 수고하셨습니다! 🎉